# Prepare Human Comments Reference

This notebook builds the cleaned human-comment reference set from `uicrit_notna_deduped.parquet`.

It produces one main output:
- `human_critiques_final.parquet`: one row per cleaned human comment set per task

These outputs are intended for later LLM-vs-human comment matching within the same task.


### **Notes on task-only deduplication**

This notebook now keeps all deduplication strictly **within the same task**. Comments from different tasks are never merged or combined.

That means the reference set preserves task-specific human feedback throughout downstream comparisons.


In [1]:
from pathlib import Path
import re

import pandas as pd

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 50)

DATA_DIR = Path("./cleaned_dataset")
INPUT_PATH = DATA_DIR / "uicrit_notna_deduped.parquet"
OUTPUT_DIR = DATA_DIR / "human_critiques"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DROP_LLM = True
DROP_MISSING_OBSERVED = True
DROP_MISSING_EXPECTED = False
DROP_MISSING_FIX = False

DEDUP_WITHIN_TASK = True

TASK_DEDUP_KEY = ["screen_task_id", "observed_issue_norm"]

INPUT_PATH

WindowsPath('cleaned_dataset/uicrit_notna_deduped.parquet')

## 1. Removing Missing and Exact Duplicate Comments

### 1.1 Load Parsed Comments

The input table is currently at screen-task level, with `parsed_comments` stored as a list of dictionaries.

In [2]:
uicrit_df = pd.read_parquet(INPUT_PATH)

print("Rows (screen-task level):", len(uicrit_df))
print("Unique screens:", uicrit_df["screen_id"].nunique())
print("Unique screen-task ids:", uicrit_df["screen_task_id"].nunique())
print("Rows with parsed comments:", uicrit_df["parsed_comments"].notna().sum())

uicrit_df[["screen_id", "app_category", "screen_task_id", "task", "parsed_comments"]].head(2)

Rows (screen-task level): 2957
Unique screens: 1000
Unique screen-task ids: 2957
Rows with parsed comments: 2957


,screen_id,app_category,screen_task_id,task,parsed_comments
0,15,Health & Fitness,15_T01,Plan and Start Full Body Workouts,"[{'bounding_box': {'x_max': 0.40545597, 'x_min': 0.1559446, 'y_max': 0.09649163, 'y_min': 0.05137866}, 'comment_label': 'Comment 1', 'comment_source': 'Huma..."
1,15,Health & Fitness,15_T02,Tap on the Plus icon or explore the Workout Plans.,"[{'bounding_box': {'x_max': 0.10555714, 'x_min': 0.02777819, 'y_max': 0.10937545, 'y_min': 0.04375018}, 'comment_label': 'Comment 1', 'comment_source': 'Hum..."


### 1.2 Flatten To Comment-Level Long Format

Each parsed comment becomes one row while preserving task provenance and its linked screen metadata.


In [3]:
base_cols = [
    "screen_id",
    "app_category",
    "screen_task_id",
    "task",
    "parsed_comments",
]

raw_comments_df = (
    uicrit_df[base_cols]
    .explode("parsed_comments", ignore_index=True)
    .rename(columns={"parsed_comments": "parsed_comment"})
)

raw_comments_df = raw_comments_df[raw_comments_df["parsed_comment"].notna()].copy()

# Preserve the original nested bounding_box dict before flattening the remaining fields.
raw_comments_df["bounding_box"] = raw_comments_df["parsed_comment"].apply(
    lambda x: x.get("bounding_box") if isinstance(x, dict) else None
)

comment_fields = raw_comments_df["parsed_comment"].apply(lambda x: x if isinstance(x, dict) else {}).apply(
    lambda d: {k: v for k, v in d.items() if k != "bounding_box"}
)
comment_fields = pd.json_normalize(comment_fields)

raw_comments_df = pd.concat(
    [raw_comments_df.drop(columns=["parsed_comment"]).reset_index(drop=True), comment_fields.reset_index(drop=True)],
    axis=1,
)

raw_comments_df.insert(0, "comment_id", [f"C_{i+1:06d}" for i in range(len(raw_comments_df))])

raw_comment_column_order = [
    "comment_id",
    "screen_id",
    "app_category",
    "screen_task_id",
    "task",
    "comment_source",
    "missing_parts",
    "expected_standard",
    "observed_issue",
    "suggested_fix",
    "bounding_box",
    "raw_text",
]
ordered_cols = [col for col in raw_comment_column_order if col in raw_comments_df.columns]
remaining_cols = [col for col in raw_comments_df.columns if col not in ordered_cols]
raw_comments_df = raw_comments_df[ordered_cols + remaining_cols].copy()

print("Comment-level rows:", len(raw_comments_df))
display(raw_comments_df.drop(columns=["comment_label"], errors="ignore").head(3))


Comment-level rows: 11365


,comment_id,screen_id,app_category,screen_task_id,task,comment_source,missing_parts,expected_standard,observed_issue,suggested_fix,bounding_box,raw_text
0,C_000001,15,Health & Fitness,15_T01,Plan and Start Full Body Workouts,Human,[],the text’s visual treatment and formatting should make it easy to understand.,the text (Workouts) is small even when it is heading.,increase font size and weight to make it look like a heading,"{'x_max': 0.40545597, 'x_min': 0.1559446, 'y_max': 0.09649163, 'y_min': 0.05137866}","Comment 1 The expected standard is that the text’s visual treatment and formatting should make it easy to understand. In the current design, the text (Worko..."
1,C_000002,15,Health & Fitness,15_T01,Plan and Start Full Body Workouts,Human,[],the text and background colors used in the design should be complementary and easy to read.,text (Plans) is in light red color on white background which is not making a good contrast.,change colors to be more complementary to each other (change texts to dark colors) to make it a good contrast and easier to read.,"{'x_max': 0.1559446, 'x_min': 0.02896114, 'y_max': 0.18045189, 'y_min': 0.15037657}","Comment 2 The expected standard is that the text and background colors used in the design should be complementary and easy to read. In the current design, t..."
2,C_000003,15,Health & Fitness,15_T01,Plan and Start Full Body Workouts,Human,[],the design should make the most important information visually dominant.,the back button size is small.,increase the back button size to make it visually prominent for the users.,"{'x_max': 0.1002501, 'x_min': 0.03787226, 'y_max': 0.09147908, 'y_min': 0.05513808}","Comment 3 The expected standard is that the design should make the most important information visually dominant. In the current design, the back button size..."


### 1.3 Initial Profiling

Before cleaning, inspect source mix, missingness, and volume.

In [4]:
raw_comments_summary = pd.Series({
    "total_comments": len(raw_comments_df),
    "unique_screens": raw_comments_df["screen_id"].nunique(),
    "unique_screen_tasks": raw_comments_df["screen_task_id"].nunique(),
    "human_comments": (raw_comments_df["comment_source"] == "Human").sum(),
    "llm_comments": (raw_comments_df["comment_source"] == "LLM").sum(),
    "missing_expected_standard": raw_comments_df["expected_standard"].isna().sum(),
    "missing_observed_issue": raw_comments_df["observed_issue"].isna().sum(),
    "missing_suggested_fix": raw_comments_df["suggested_fix"].isna().sum(),
})

raw_comments_summary.to_frame("count")


,count
total_comments,11365
unique_screens,1000
unique_screen_tasks,2954
human_comments,8277
llm_comments,3088
missing_expected_standard,12
missing_observed_issue,7
missing_suggested_fix,17


In [5]:
comments_per_task = raw_comments_df.groupby("screen_task_id").size().rename("num_comments")
comments_per_screen = raw_comments_df.groupby("screen_id").size().rename("num_comments")

print("Comments per task")
print(comments_per_task.describe())
print()
print("Comments per screen")
print(comments_per_screen.describe())


Comments per task
count    2954.000000
mean        3.847326
std         1.924774
min         1.000000
25%         2.000000
50%         4.000000
75%         5.000000
max        13.000000
Name: num_comments, dtype: float64

Comments per screen
count    1000.000000
mean       11.365000
std         3.654193
min         3.000000
25%         9.000000
50%        11.000000
75%        14.000000
max        27.000000
Name: num_comments, dtype: float64


In [6]:
all_task_ids = set(uicrit_df["screen_task_id"].dropna().astype(str))
exploded_task_ids = set(raw_comments_df["screen_task_id"].dropna().astype(str))

missing_task_ids = sorted(all_task_ids - exploded_task_ids)

print("Number of screen_task_id values missing from raw_comments_df:", len(missing_task_ids))
print("Missing screen_task_id values:")
print(missing_task_ids)

missing_rows = (
    uicrit_df[uicrit_df["screen_task_id"].astype(str).isin(missing_task_ids)]
    .loc[:, ["screen_id", "screen_task_id", "task", "parsed_comments"]]
    .sort_values(["screen_id", "screen_task_id"])
    .reset_index(drop=True)
)

display(missing_rows)


Number of screen_task_id values missing from raw_comments_df: 3
Missing screen_task_id values:
['26804_T03', '27590_T02', '58424_T01']


,screen_id,screen_task_id,task,parsed_comments
0,26804,26804_T03,choose the Pokemon from the list,[]
1,27590,27590_T02,Select a template,[]
2,58424,58424_T01,Login using Google or Facebook,[]


### 1.4 Normalize Text Fields

Normalization supports exact deduplication and later matching. Keep both original and normalized forms.

In [7]:
TEXT_FIELDS = ["expected_standard", "observed_issue", "suggested_fix", "raw_text"]

def normalize_text(value):
    if value is None:
        return None
    if isinstance(value, float) and pd.isna(value):
        return None

    text = str(value).strip()
    if not text:
        return None

    text = text.lower()
    text = text.replace("’", "'")
    text = text.replace("‘", "'")
    text = text.replace("“", '"')
    text = text.replace("”", '"')
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"^[^\w]+|[^\w]+$", "", text)
    return text or None

for field in TEXT_FIELDS:
    raw_comments_df[f"{field}_norm"] = raw_comments_df[field].apply(normalize_text)

raw_comments_df["missing_expected_standard"] = raw_comments_df["expected_standard_norm"].isna()
raw_comments_df["missing_observed_issue"] = raw_comments_df["observed_issue_norm"].isna()
raw_comments_df["missing_suggested_fix"] = raw_comments_df["suggested_fix_norm"].isna()

raw_comments_df[[
    "comment_id",
    "screen_id",
    "screen_task_id",
    "comment_source",
    "observed_issue",
    "observed_issue_norm",
]].head(5)


,comment_id,screen_id,screen_task_id,comment_source,observed_issue,observed_issue_norm
0,C_000001,15,15_T01,Human,the text (Workouts) is small even when it is heading.,the text (workouts) is small even when it is heading
1,C_000002,15,15_T01,Human,text (Plans) is in light red color on white background which is not making a good contrast.,text (plans) is in light red color on white background which is not making a good contrast
2,C_000003,15,15_T01,Human,the back button size is small.,the back button size is small
3,C_000004,15,15_T01,Human,the texts are disappearing at the bottom edge of the layout leaving no marginal space which is making it difficult for users to know the complete information,the texts are disappearing at the bottom edge of the layout leaving no marginal space which is making it difficult for users to know the complete information
4,C_000005,15,15_T01,Human,Icon is not clearly visible,icon is not clearly visible


### 1.5 Apply Cleaning Rules

Hard removals are logged in an audit table rather than being silently discarded.

In [8]:
audit_parts = []
filtered_human_comments_df = raw_comments_df.copy()

def move_to_audit(df, mask, reason):
    removed = df.loc[mask].copy()
    if len(removed):
        removed["audit_reason"] = reason
    kept = df.loc[~mask].copy()
    return kept, removed

if DROP_LLM:
    filtered_human_comments_df, removed = move_to_audit(
        filtered_human_comments_df,
        filtered_human_comments_df["comment_source"].ne("Human"),
        "non_human_comment",
    )
    audit_parts.append(removed)

if DROP_MISSING_OBSERVED:
    filtered_human_comments_df, removed = move_to_audit(
        filtered_human_comments_df,
        filtered_human_comments_df["missing_observed_issue"],
        "missing_observed_issue",
    )
    audit_parts.append(removed)

if DROP_MISSING_EXPECTED:
    filtered_human_comments_df, removed = move_to_audit(
        filtered_human_comments_df,
        filtered_human_comments_df["missing_expected_standard"],
        "missing_expected_standard",
    )
    audit_parts.append(removed)

if DROP_MISSING_FIX:
    filtered_human_comments_df, removed = move_to_audit(
        filtered_human_comments_df,
        filtered_human_comments_df["missing_suggested_fix"],
        "missing_suggested_fix",
    )
    audit_parts.append(removed)

removed_comments_audit_df = (
    pd.concat(audit_parts, ignore_index=True)
    if audit_parts
    else pd.DataFrame(columns=list(filtered_human_comments_df.columns) + ["audit_reason"])
)

print("Remaining after hard filters:", len(filtered_human_comments_df))
print("Removed by hard filters:", len(removed_comments_audit_df))
removed_comments_audit_df["audit_reason"].value_counts(dropna=False)


Remaining after hard filters: 8272
Removed by hard filters: 3093


audit_reason
non_human_comment         3088
missing_observed_issue       5
Name: count, dtype: int64

### 1.6 Deduplicate Within The Same Task

First, drop exact duplicate triplets within the same `screen_task_id` using normalized
`expected_standard`, `observed_issue`, and `suggested_fix`.

For rows that still repeat only on normalized `observed_issue`, review them manually one group at a time
and choose which row to keep.


In [9]:
TASK_TRIPLET_KEY = [
    "screen_task_id",
    "expected_standard_norm",
    "observed_issue_norm",
    "suggested_fix_norm",
]

task_dedup_comments_df = filtered_human_comments_df.copy().sort_values(
    ["screen_task_id", "comment_label", "comment_id"],
    kind="stable",
)

task_dedup_comments_df["duplicate_task_triplet"] = task_dedup_comments_df.duplicated(
    subset=TASK_TRIPLET_KEY,
    keep="first",
)
task_triplet_duplicates_df = task_dedup_comments_df[task_dedup_comments_df["duplicate_task_triplet"]].copy()
task_triplet_duplicates_df["audit_reason"] = "duplicate_task_triplet"

task_comments_after_triplet_df = task_dedup_comments_df[~task_dedup_comments_df["duplicate_task_triplet"]].copy()

task_duplicate_review_df = task_comments_after_triplet_df[
    task_comments_after_triplet_df.duplicated(subset=TASK_DEDUP_KEY, keep=False)
].copy().sort_values(["screen_task_id", "observed_issue_norm", "comment_id"], kind="stable")

task_duplicate_groups = list(task_duplicate_review_df.groupby(TASK_DEDUP_KEY, dropna=False).groups.keys())

print("Task-level rows before dedup:", len(task_dedup_comments_df))
print("Exact triplet duplicates removed within task:", len(task_triplet_duplicates_df))
print("Observed-issue duplicate groups for manual review within task:", len(task_duplicate_groups))

task_reference_comments_df = task_comments_after_triplet_df.copy()


Task-level rows before dedup: 8272
Exact triplet duplicates removed within task: 47
Observed-issue duplicate groups for manual review within task: 22


In [10]:
def show_task_duplicate_group(group_idx):
    if group_idx < 0 or group_idx >= len(task_duplicate_groups):
        raise IndexError(f"group_idx must be between 0 and {len(task_duplicate_groups) - 1}")

    screen_task_id, observed_issue_norm = task_duplicate_groups[group_idx]
    group_df = task_duplicate_review_df[
        (task_duplicate_review_df["screen_task_id"] == screen_task_id)
        & (task_duplicate_review_df["observed_issue_norm"] == observed_issue_norm)
    ].copy()

    print(f"Task duplicate group {group_idx + 1}/{len(task_duplicate_groups)}")
    print("screen_task_id:", screen_task_id)
    print("observed_issue_norm:", observed_issue_norm)
    return group_df[[
        "comment_id",
        "screen_id",
        "screen_task_id",
        "comment_source",
        "expected_standard",
        "observed_issue",
        "suggested_fix",
        "missing_parts",
    ]]

def apply_task_manual_choices(keep_comment_ids):
    if len(task_duplicate_groups) == 0:
        return task_comments_after_triplet_df.copy()

    if len(keep_comment_ids) != len(task_duplicate_groups):
        raise ValueError(f"Provide exactly one comment_id per task duplicate group ({len(task_duplicate_groups)} total).")

    chosen = set(keep_comment_ids)
    duplicate_mask = task_comments_after_triplet_df.duplicated(subset=TASK_DEDUP_KEY, keep=False)
    unresolved_group_rows = task_comments_after_triplet_df.loc[duplicate_mask, "comment_id"]

    if not set(keep_comment_ids).issubset(set(unresolved_group_rows)):
        raise ValueError("All chosen comment_id values must come from task duplicate groups.")

    kept_duplicate_rows = task_comments_after_triplet_df[
        duplicate_mask & task_comments_after_triplet_df["comment_id"].isin(chosen)
    ].copy()
    if len(kept_duplicate_rows) != len(task_duplicate_groups):
        raise ValueError("Your selections do not resolve to exactly one kept row per task duplicate group.")

    return pd.concat([
        task_comments_after_triplet_df[~duplicate_mask].copy(),
        kept_duplicate_rows,
    ], ignore_index=True).sort_values(["screen_task_id", "comment_label", "comment_id"], kind="stable").reset_index(drop=True)

print("Use show_task_duplicate_group(i) to inspect one group at a time.")
print("Then run: task_reference_comments_df = apply_task_manual_choices([...])")
print("Example first group:")
show_task_duplicate_group(0) if task_duplicate_groups else print("No task-level observed-issue duplicate groups to review.")


Use show_task_duplicate_group(i) to inspect one group at a time.
Then run: task_reference_comments_df = apply_task_manual_choices([...])
Example first group:
Task duplicate group 1/22
screen_task_id: 1395_T02
observed_issue_norm: the highlighted heading is not visually prominent


,comment_id,screen_id,screen_task_id,comment_source,expected_standard,observed_issue,suggested_fix,missing_parts
275,C_000276,1395,1395_T02,Human,make the most important information visually dominant.,the highlighted heading is not visually prominent.,we can enlarge it.,[]
276,C_000277,1395,1395_T02,Human,make the most important information visually dominant.,the highlighted heading is not visually prominent.,we can enlarge the heading.,[]


In [11]:
# After manually inspecting the groups, most parts were very similar with slight variations in wording. I chose the more rich and representative comment from each group.

task_reference_comments_df = apply_task_manual_choices([
    "C_000277", "C_002578", "C_002580", "C_000382", "C_003524", "C_003605",
    "C_004024", "C_004572", "C_005400", "C_005675", "C_006273", "C_006625",
    "C_007188", "C_007930", "C_000825", "C_008399", "C_009220", "C_009570",
    "C_010230", "C_011218", "C_001524", "C_001557"
])

### 1.7 Build Final Task-Shaped Output

The final saved table is reshaped back to one row per task with a `comments` list of dictionaries:
`screen_id`, `app_category`, `screen_task_id`, `task`, `comments`.


In [12]:
task_metadata_df = (
    uicrit_df[["screen_id", "app_category", "screen_task_id", "task"]]
    .drop_duplicates(subset=["screen_task_id"])
    .copy()
)

comment_payload_cols = [
    "comment_label",
    "missing_parts",
    "expected_standard",
    "observed_issue",
    "suggested_fix",
    "bounding_box",
    "raw_text",
]

final_human_reference_df = (
    task_reference_comments_df
    .sort_values(["screen_id", "screen_task_id", "comment_label"], kind="stable")
    .groupby(["screen_id", "screen_task_id", "task"], sort=True)[comment_payload_cols]
    .apply(lambda x: x.to_dict(orient="records"))
    .reset_index(name="comments")
)

final_human_reference_df = (
    task_metadata_df.merge(final_human_reference_df, on=["screen_id", "screen_task_id", "task"], how="left")
    .sort_values(["screen_id", "screen_task_id"], kind="stable")
    .reset_index(drop=True)
)

final_human_reference_df["comments"] = final_human_reference_df["comments"].apply(lambda x: x if isinstance(x, list) else [])
final_human_reference_before_empty_drop_df = final_human_reference_df.copy()

removed_empty_comment_tasks = int((final_human_reference_before_empty_drop_df["comments"].apply(len) == 0).sum())
original_num_screen_tasks = len(final_human_reference_before_empty_drop_df)

final_human_reference_df = final_human_reference_before_empty_drop_df[
    final_human_reference_before_empty_drop_df["comments"].apply(len) > 0
].copy().reset_index(drop=True)
final_human_reference_df = final_human_reference_df[["screen_id", "app_category", "screen_task_id", "task", "comments"]]

print("Original screen_task rows:", original_num_screen_tasks)
print("Removed screen_task rows with empty comments:", removed_empty_comment_tasks)
print("Final screen_task rows:", len(final_human_reference_df))

display(
    final_human_reference_df.assign(num_comments=final_human_reference_df["comments"].apply(len))
    [["screen_id", "app_category", "screen_task_id", "task", "num_comments"]]
    .head(3)
)

Original screen_task rows: 2957
Removed screen_task rows with empty comments: 970
Final screen_task rows: 1987


,screen_id,app_category,screen_task_id,task,num_comments
0,15,Health & Fitness,15_T01,Plan and Start Full Body Workouts,9
1,15,Health & Fitness,15_T02,Tap on the Plus icon or explore the Workout Plans.,5
2,28,Finance,28_T02,Enter details to sign in or click on activate mobile banking/Need help signing in.,5


#### Final Cleaning Check

Inspect which screen-task rows are removed at the final step because their `comments` list is empty.
Interpretation of why they became empty should rely on the earlier filtering and audit tables.


In [13]:
final_reference_with_counts_df = final_human_reference_before_empty_drop_df[["screen_id", "app_category", "screen_task_id", "task", "comments"]].copy()
final_reference_with_counts_df["num_comments"] = final_reference_with_counts_df["comments"].apply(len)

empty_comment_tasks_df = final_reference_with_counts_df[final_reference_with_counts_df["num_comments"] == 0].copy()

print("Original screen-task rows before final empty-comments drop:", original_num_screen_tasks)
print("Removed screen-task rows with empty comments:", removed_empty_comment_tasks)
print("Final retained screen-task rows:", len(final_human_reference_df))

display(empty_comment_tasks_df.head(10))

Original screen-task rows before final empty-comments drop: 2957
Removed screen-task rows with empty comments: 970
Final retained screen-task rows: 1987


,screen_id,app_category,screen_task_id,task,comments,num_comments
2,15,Health & Fitness,15_T03,select and get started with Full body workouts,[],0
3,28,Finance,28_T01,Enter details to Sing In to Scotiabank.,[],0
8,67,Education,67_T03,View/ Upload photo.,[],0
12,193,Social,193_T03,Search members to make new friends,[],0
13,233,Weather,233_T01,Change Map Settings.,[],0
18,288,Social,288_T03,Select your preferences.,[],0
21,342,None,342_T03,Send Feedback about Great Sword.,[],0
24,422,Social,422_T03,sign-in as Laura and edit the details,[],0
26,445,Medical,445_T02,Select to add the premium feature,[],0
29,640,Music & Audio,640_T02,Choose options to listen FM radio stations,[],0


### 1.8 Validation Checks

These checks confirm that the final task-shaped table is ready to save.


In [14]:
assert (task_reference_comments_df["comment_source"] == "Human").all(), "Non-human comment survived task-level cleaning."
assert task_reference_comments_df["observed_issue"].notna().all(), "Task-level reference contains missing observed issue."
assert not task_reference_comments_df.duplicated(subset=TASK_DEDUP_KEY).any(), "Task-level observed-issue duplicates still remain. Resolve them manually first."
assert final_human_reference_df.columns.tolist() == ["screen_id", "app_category", "screen_task_id", "task", "comments"], "Final output columns do not match the expected schema."
assert final_human_reference_df["screen_task_id"].is_unique, "Final output should have one row per screen_task_id."
assert final_human_reference_df["comments"].apply(len).gt(0).all(), "Final output still contains empty comments lists."

print("All validation checks passed.")


All validation checks passed.


### 1.9 Summary

Summarize both what was dropped during cleaning and what remains in the final dataset.


In [15]:
final_comment_counts = final_human_reference_df["comments"].apply(len)

final_human_comments_df = final_human_reference_df[["screen_id", "app_category", "screen_task_id", "task", "comments"]].explode("comments", ignore_index=True)
final_human_comments_df = final_human_comments_df[final_human_comments_df["comments"].notna()].copy()

if len(final_human_comments_df):
    final_comment_fields = pd.json_normalize(final_human_comments_df["comments"])
    final_human_comments_df = pd.concat(
        [final_human_comments_df.drop(columns=["comments"]).reset_index(drop=True), final_comment_fields.reset_index(drop=True)],
        axis=1,
    )
else:
    final_human_comments_df = pd.DataFrame(columns=["screen_id", "app_category", "screen_task_id", "task"])

final_comment_column_order = [
    "comment_id",
    "screen_id",
    "app_category",
    "screen_task_id",
    "task",
    "comment_source",
    "missing_parts",
    "expected_standard",
    "observed_issue",
    "suggested_fix",
    "bounding_box",
    "raw_text",
]
ordered_cols = [col for col in final_comment_column_order if col in final_human_comments_df.columns]
remaining_cols = [col for col in final_human_comments_df.columns if col not in ordered_cols]
final_human_comments_df = final_human_comments_df[ordered_cols + remaining_cols].copy()

cleaning_summary_df = pd.DataFrame([
    {"metric": "removed_non_human_comments", "value": int((removed_comments_audit_df["audit_reason"] == "non_human_comment").sum())},
    {"metric": "removed_missing_observed_issue", "value": int((removed_comments_audit_df["audit_reason"] == "missing_observed_issue").sum())},
    {"metric": "removed_duplicate_task_triplet", "value": int(len(task_triplet_duplicates_df))},
    {"metric": "task_duplicate_groups_manually_reviewed", "value": int(len(task_duplicate_groups))},
])

display(cleaning_summary_df)


,metric,value
0,removed_non_human_comments,3088
1,removed_missing_observed_issue,5
2,removed_duplicate_task_triplet,47
3,task_duplicate_groups_manually_reviewed,22


In [29]:
num_missing_expected = final_human_comments_df["expected_standard"].isna().sum() if "expected_standard" in final_human_comments_df.columns else 0
num_missing_fix = final_human_comments_df["suggested_fix"].isna().sum() if "suggested_fix" in final_human_comments_df.columns else 0
num_missing_both = ((final_human_comments_df["expected_standard"].isna()) & (final_human_comments_df["suggested_fix"].isna())).sum() if len(final_human_comments_df) else 0

descriptive_summary = pd.DataFrame([
    {"metric": "original_num_screen_tasks", "value": int(original_num_screen_tasks)},
    {"metric": "removed_empty_comment_screen_tasks", "value": int(removed_empty_comment_tasks)},
    {"metric": "final_num_screen_tasks", "value": int(len(final_human_reference_df))},
    {"metric": "num_screens", "value": int(final_human_reference_df["screen_id"].nunique())},
    {"metric": "num_final_comments", "value": int(len(final_human_comments_df))},
    {"metric": "avg_comments_per_task", "value": float(final_comment_counts.mean())},
    {"metric": "median_comments_per_task", "value": float(final_comment_counts.median())},
    {"metric": "missing_expected_standard", "value": int(num_missing_expected)},
    {"metric": "missing_suggested_fix", "value": int(num_missing_fix)},
    {"metric": "missing_expected_and_fix", "value": int(num_missing_both)},
])

display(descriptive_summary.round(0))
print("Comments per retained task distribution:")
print(final_comment_counts.describe().round(1))

,metric,value
0,original_num_screen_tasks,2957.0
1,removed_empty_comment_screen_tasks,970.0
2,final_num_screen_tasks,1987.0
3,num_screens,1000.0
4,num_final_comments,8203.0
5,avg_comments_per_task,4.0
6,median_comments_per_task,4.0
7,missing_expected_standard,4.0
8,missing_suggested_fix,6.0
9,missing_expected_and_fix,1.0


Comments per retained task distribution:
count    1987.0
mean        4.1
std         2.1
min         1.0
25%         2.0
50%         4.0
75%         6.0
max        13.0
Name: comments, dtype: float64


### 1.10 Save Outputs

Save only the parquet outputs needed for downstream analysis.


In [17]:
final_human_reference_df_path = DATA_DIR / "human_comments_exact_dedup.parquet"
audit_path = OUTPUT_DIR / "human_comments_exact_dedup_audit_log.parquet"

final_human_reference_df.to_parquet(final_human_reference_df_path, index=False)
removed_comments_audit_df.to_parquet(audit_path, index=False)

print("Saved:")
for path in [final_human_reference_df_path, audit_path]:
    print(path)

Saved:
cleaned_dataset\human_comments_exact_dedup.parquet
cleaned_dataset\human_critiques\human_comments_exact_dedup_audit_log.parquet


## 2. Semantic Duplicates Within The Same Task

Workflow:
- generate candidate comment pairs only within the same `screen_task_id`
- first score candidate pairs with embeddings + cosine similarity using a relatively high threshold
- manually inspect a sample of high-similarity pairs to tune the threshold
- give a summary of the duplications
- send the remaining candidate pairs to OpenAI Batch for a stricter same-meaning judgment
- keep one canonical row per semantic group after review


### 2.0 Imports And Setup

Load the shared semantic-matching utilities and batch helpers used throughout Section 2.


In [1]:
import sys
from pathlib import Path

import pandas as pd
from openai import OpenAI

DATA_DIR = Path("./cleaned_dataset")
OUTPUT_DIR = DATA_DIR / "human_critiques"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT = Path.cwd().resolve().parents[1]
sys.path.append(str(PROJECT_ROOT / "code"))

from utils.comment_matching_utils import (
    DEFAULT_EMBEDDING_MODEL_NAME,
    DEFAULT_REVIEW_MODEL,
    build_standard_preview_df,
    build_standardized_human_comments_from_nested,
    build_standardized_pairs,
    identify_review_needed,
    inspect_manual_review_rows,
    finalize_review_decisions,
    load_embedding_model,
    parse_semantic_review_results_jsonl,
    build_semantic_review_requests,
    score_standardized_pairs,
    write_jsonl,
)
from utils.openai_batch_manager import OpenAIBatchManager

final_human_reference_df_path = DATA_DIR / "human_comments_exact_dedup.parquet"

c:\Users\mahmo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 2.1 Build Comment-Level Input

Start from `final_human_reference_df` and explode the `comments` column so each cleaned human comment is one row.
Keep the screen/task identifiers needed to map decisions back later.


In [2]:
final_human_reference_df = pd.read_parquet(final_human_reference_df_path)

semantic_comments_df = build_standardized_human_comments_from_nested(
    final_human_reference_df[["screen_id", "app_category", "screen_task_id", "task", "comments"]].copy(),
    nested_comments_col="comments",
    comment_id_prefix="H",
)

print("Comment-level rows for semantic dedup:", len(semantic_comments_df))
print("Tasks represented:", semantic_comments_df["screen_task_id"].nunique())
display(semantic_comments_df.drop(columns=["comment_label"], errors="ignore").head(3))

Comment-level rows for semantic dedup: 8203
Tasks represented: 1987


,comment_id,screen_id,app_category,screen_task_id,task,source_type,expected_standard,observed_issue,suggested_fix,match_text,missing_parts,bounding_box,raw_text
0,H_000001,15,Health & Fitness,15_T01,Plan and Start Full Body Workouts,human,the text’s visual treatment and formatting sho...,the text (Workouts) is small even when it is h...,increase font size and weight to make it look...,the text (Workouts) is small even when it is h...,[],"{'x_max': 0.40545597, 'x_min': 0.1559446, 'y_m...",Comment 1 The expected standard is that the te...
1,H_000002,15,Health & Fitness,15_T01,Plan and Start Full Body Workouts,human,the text and background colors used in the des...,text (Plans) is in light red color on white ba...,change colors to be more complementary to each...,text (Plans) is in light red color on white ba...,[],"{'x_max': 0.1559446, 'x_min': 0.02896114, 'y_m...",Comment 2 The expected standard is that the te...
2,H_000003,15,Health & Fitness,15_T01,Plan and Start Full Body Workouts,human,the design should make the most important info...,the back button size is small.,increase the back button size to make it visua...,the back button size is small.,[],"{'x_max': 0.1002501, 'x_min': 0.03787226, 'y_m...",Comment 3 The expected standard is that the de...


### 2.2 Generate Within-Task Candidate Pairs

Create candidate pairs only within the same `screen_task_id`.
Do not compare comments across different tasks.


In [3]:
semantic_comment_pairs_df = build_standardized_pairs(
    semantic_comments_df,
    semantic_comments_df,
    pair_id_prefix="PAIR",
    exclude_same_comment_id_pairs=True,
    group_col="screen_task_id",
)
semantic_comment_pairs_df = semantic_comment_pairs_df[
    semantic_comment_pairs_df["left_comment_id"] < semantic_comment_pairs_df["right_comment_id"]
].copy().reset_index(drop=True)

print("Within-task candidate pairs:", len(semantic_comment_pairs_df))
print("Tasks with at least one pair:", semantic_comment_pairs_df["screen_task_id"].nunique() if len(semantic_comment_pairs_df) else 0)

if len(semantic_comment_pairs_df):
    pairs_per_task = semantic_comment_pairs_df.groupby("screen_task_id").size()
    print("Candidate pairs per task:")
    print(pairs_per_task.describe().round(2))

semantic_comment_pairs_df.head(5)


Within-task candidate pairs: 17085
Tasks with at least one pair: 1840
Candidate pairs per task:
count    1840.00
mean        9.29
std         9.25
min         1.00
25%         3.00
50%         6.00
75%        15.00
max        78.00
dtype: float64


,pair_id,screen_task_id,app_category,left_comment_id,left_source_type,left_task,left_observed_issue,left_expected_standard,left_suggested_fix,left_match_text,...,right_task,right_observed_issue,right_expected_standard,right_suggested_fix,right_match_text,missing_parts_right,bounding_box_right,raw_text_right,comment_label_right,screen_id
0,PAIR_0000001,15_T01,Health & Fitness,H_000001,human,Plan and Start Full Body Workouts,the text (Workouts) is small even when it is h...,the text’s visual treatment and formatting sho...,increase font size and weight to make it look...,the text (Workouts) is small even when it is h...,...,Plan and Start Full Body Workouts,text (Plans) is in light red color on white ba...,the text and background colors used in the des...,change colors to be more complementary to each...,text (Plans) is in light red color on white ba...,[],"{'x_max': 0.1559446, 'x_min': 0.02896114, 'y_m...",Comment 2 The expected standard is that the te...,Comment 2,15
1,PAIR_0000002,15_T01,Health & Fitness,H_000001,human,Plan and Start Full Body Workouts,the text (Workouts) is small even when it is h...,the text’s visual treatment and formatting sho...,increase font size and weight to make it look...,the text (Workouts) is small even when it is h...,...,Plan and Start Full Body Workouts,the back button size is small.,the design should make the most important info...,increase the back button size to make it visua...,the back button size is small.,[],"{'x_max': 0.1002501, 'x_min': 0.03787226, 'y_m...",Comment 3 The expected standard is that the de...,Comment 3,15
2,PAIR_0000003,15_T01,Health & Fitness,H_000001,human,Plan and Start Full Body Workouts,the text (Workouts) is small even when it is h...,the text’s visual treatment and formatting sho...,increase font size and weight to make it look...,the text (Workouts) is small even when it is h...,...,Plan and Start Full Body Workouts,the texts are disappearing at the bottom edge ...,design should be well organized.,redesign the UI to fit the elements within the...,the texts are disappearing at the bottom edge ...,[],"{'x_max': 0.75029596, 'x_min': 0.05138287, 'y_...",Comment 4 The expected standard is that design...,Comment 4,15
3,PAIR_0000004,15_T01,Health & Fitness,H_000001,human,Plan and Start Full Body Workouts,the text (Workouts) is small even when it is h...,the text’s visual treatment and formatting sho...,increase font size and weight to make it look...,the text (Workouts) is small even when it is h...,...,Plan and Start Full Body Workouts,Icon is not clearly visible,the design elements should be clearly visible,change icon color to dark color to make it cle...,Icon is not clearly visible,[],"{'x_max': 0.90002314, 'x_min': 0.81536749, 'y_...",Comment 5 The expected standard is that the de...,Comment 5,15
4,PAIR_0000005,15_T01,Health & Fitness,H_000001,human,Plan and Start Full Body Workouts,the text (Workouts) is small even when it is h...,the text’s visual treatment and formatting sho...,increase font size and weight to make it look...,the text (Workouts) is small even when it is h...,...,Plan and Start Full Body Workouts,"""Plus"" sign overlaps the other elements (Icon)",every element should have some connection to a...,redesign the page to make the other elements c...,"""Plus"" sign overlaps the other elements (Icon)",[],"{'x_max': 0.98022322, 'x_min': 0.81091193, 'y_...",Comment 6 The expected standard is that every ...,Comment 6,15


### 2.3 Embeddings And Cosine Similarity

Embed each comment text, compute cosine similarity for the within-task pairs, and keep only pairs above a relatively high threshold.
Use this as a conservative semantic-duplicate candidate stage, not as the final decision.


In [4]:
EMBEDDING_MODEL_NAME = DEFAULT_EMBEDDING_MODEL_NAME

embedding_model = load_embedding_model(EMBEDDING_MODEL_NAME)
scored_semantic_pairs_df, semantic_embeddings_df = score_standardized_pairs(
    semantic_comment_pairs_df,
    semantic_comments_df,
    embedding_model=embedding_model,
)

print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Embedding rows:", len(semantic_embeddings_df))
print("Candidate pairs scored:", len(scored_semantic_pairs_df))
print("Cosine similarity distribution:")
print(scored_semantic_pairs_df["cosine_similarity"].describe().round(4))


Embedding model: all-mpnet-base-v2
Embedding rows: 8203
Candidate pairs scored: 17085
Cosine similarity distribution:
count    17085.0000
mean         0.3754
std          0.1843
min         -0.1003
25%          0.2472
50%          0.3576
75%          0.4811
max          0.9888
Name: cosine_similarity, dtype: float64


In [5]:
SEMANTIC_SIMILARITY_THRESHOLD = 0.90

high_similarity_semantic_pairs_df = scored_semantic_pairs_df[
    scored_semantic_pairs_df["cosine_similarity"] >= SEMANTIC_SIMILARITY_THRESHOLD
].copy().sort_values(["cosine_similarity", "screen_task_id"], ascending=[False, True], kind="stable")

linked_pair_comment_lookup_df = semantic_comments_df[[
    "comment_id",
    "screen_id",
    "screen_task_id",
    "source_type",
    "observed_issue",
]].copy()

parent = {}

def find(x):
    parent.setdefault(x, x)
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x

def union(a, b):
    ra = find(a)
    rb = find(b)
    if ra != rb:
        parent[rb] = ra

for row in high_similarity_semantic_pairs_df.itertuples(index=False):
    union(row.left_comment_id, row.right_comment_id)

linked_component_rows = []
for row in high_similarity_semantic_pairs_df.itertuples(index=False):
    component_id = find(row.left_comment_id)
    linked_component_rows.append({
        "pair_id": row.pair_id,
        "screen_id": row.screen_id,
        "screen_task_id": row.screen_task_id,
        "component_id": component_id,
        "left_comment_id": row.left_comment_id,
        "right_comment_id": row.right_comment_id,
        "cosine_similarity": row.cosine_similarity,
    })

linked_pair_edges_df = pd.DataFrame(linked_component_rows)

if len(linked_pair_edges_df):
    linked_pair_group_summary_df = (
        linked_pair_edges_df
        .groupby(["screen_id", "screen_task_id", "component_id"], as_index=False)
        .agg(
            num_pairs=("pair_id", "size"),
            max_cosine_similarity=("cosine_similarity", "max"),
            min_cosine_similarity=("cosine_similarity", "min"),
        )
        .sort_values(["num_pairs", "max_cosine_similarity", "screen_task_id"], ascending=[False, False, True], kind="stable")
        .reset_index(drop=True)
    )

    component_members = {}
    for row in linked_pair_edges_df.itertuples(index=False):
        key = (row.screen_id, row.screen_task_id, row.component_id)
        component_members.setdefault(key, set()).update([row.left_comment_id, row.right_comment_id])

    linked_pair_group_summary_df["comment_ids"] = linked_pair_group_summary_df.apply(
        lambda row: sorted(component_members[(row.screen_id, row.screen_task_id, row.component_id)]),
        axis=1,
    )
    linked_pair_group_summary_df["num_comments"] = linked_pair_group_summary_df["comment_ids"].apply(len)
else:
    linked_pair_group_summary_df = pd.DataFrame(columns=[
        "screen_id", "screen_task_id", "component_id", "num_pairs", "max_cosine_similarity",
        "min_cosine_similarity", "comment_ids", "num_comments"
    ])

print("Pairs above threshold:", len(high_similarity_semantic_pairs_df))
print("Threshold:", SEMANTIC_SIMILARITY_THRESHOLD)
print("Linked groups with at least one above-threshold pair:", len(linked_pair_group_summary_df))

display(
    build_standard_preview_df(
        high_similarity_semantic_pairs_df,
        include_score=True,
        limit=10,
        sort_cols=["cosine_similarity", "screen_task_id"],
        ascending=[True, True],
    )
)

group_display_cols = [
    "screen_id",
    "screen_task_id",
    "component_id",
    "num_comments",
    "num_pairs",
    "max_cosine_similarity",
    "min_cosine_similarity",
    "comment_ids",
]
display(linked_pair_group_summary_df[group_display_cols].head(20))


def show_linked_pair_group(group_idx: int):
    if group_idx < 0 or group_idx >= len(linked_pair_group_summary_df):
        raise IndexError(f"group_idx must be between 0 and {len(linked_pair_group_summary_df) - 1}")

    group_row = linked_pair_group_summary_df.iloc[group_idx]
    member_ids = set(group_row.comment_ids)

    member_comments_df = (
        linked_pair_comment_lookup_df[linked_pair_comment_lookup_df["comment_id"].isin(member_ids)]
        .drop(columns=["comment_label"], errors="ignore")
        .sort_values(["screen_task_id", "comment_id"], kind="stable")
        .reset_index(drop=True)
    )
    member_edges_df = (
        linked_pair_edges_df[
            (linked_pair_edges_df["screen_id"] == group_row.screen_id)
            & (linked_pair_edges_df["screen_task_id"] == group_row.screen_task_id)
            & (linked_pair_edges_df["component_id"] == group_row.component_id)
        ]
        .sort_values(["cosine_similarity", "pair_id"], ascending=[False, True], kind="stable")
        .reset_index(drop=True)
    )

    display(pd.DataFrame([group_row[group_display_cols].to_dict()]))
    display(member_comments_df)
    display(member_edges_df)

Pairs above threshold: 87
Threshold: 0.9
Linked groups with at least one above-threshold pair: 85


,screen_id,screen_task_id,left_observed_issue,right_observed_issue,cosine_similarity
496,1741,1741_T01,"the text font size is small, and the backgroun...",the text is difficult to read because it is to...,0.900590
506,1741,1741_T03,"the text is small and difficult to read, and t...",the text is difficult to read because it is to...,0.901120
14840,62872,62872_T01,"the text is small and difficult to read, and t...",the text is difficult to read because it is to...,0.901120
13973,58952,58952_T01,the text (15) is too small and difficult to read.,The text is too small and hard to read.,0.901264
11290,47284,47284_T03,the font color is too light and that is why it...,the text is difficult to read because it is to...,0.901589
15000,63397,63397_T01,the text is not readable.,the text is difficult to read.,0.902096
7734,33470,33470_T02,the text's small size and low color contrast w...,"the text is small and difficult to read, and t...",0.902336
9278,39384,39384_T01,the cluttered layout impedes readability and c...,"the layout is cluttered, making it difficult t...",0.902674
5559,24345,24345_T02,"there is an error in spelling ""hight"" should b...","the word ""high"" is misspelled as ""hight"".",0.903404
2084,8321,8321_T03,the small font size and dense text make it dif...,the font is too small and the text is too dens...,0.903456


,screen_id,screen_task_id,component_id,num_comments,num_pairs,max_cosine_similarity,min_cosine_similarity,comment_ids
0,16484,16484_T01,H_001989,3,3,0.979302,0.950505,"[H_001989, H_001990, H_001992]"
1,71747,71747_T01,H_008169,2,1,0.988805,0.988805,"[H_008169, H_008173]"
2,12175,12175_T01,H_001477,2,1,0.985627,0.985627,"[H_001477, H_001478]"
3,34546,34546_T02,H_003877,2,1,0.981300,0.981300,"[H_003877, H_003879]"
4,27321,27321_T01,H_003052,2,1,0.978443,0.978443,"[H_003052, H_003054]"
5,23441,23441_T03,H_002619,2,1,0.977479,0.977479,"[H_002619, H_002624]"
6,27748,27748_T01,H_003122,2,1,0.975289,0.975289,"[H_003122, H_003123]"
7,422,422_T01,H_000071,2,1,0.975289,0.975289,"[H_000071, H_000072]"
8,43047,43047_T03,H_004868,2,1,0.975289,0.975289,"[H_004868, H_004869]"
9,62693,62693_T03,H_007106,2,1,0.975289,0.975289,"[H_007106, H_007107]"


### 2.4 Apply Simple Pairwise Threshold Dedup

Use a simple pairwise rule for comments above the cosine-similarity threshold.

Rule:
1. Treat each above-threshold pair independently.
2. Compare the two comments by available text length.
3. Keep the longer comment and mark the shorter one for removal.
4. If the lengths tie, keep the lexicographically smaller `comment_id`.
5. Rebuild the task-shaped dataframe from the comments that were not marked for removal.


In [6]:
semantic_pairwise_comments_df = semantic_comments_df.copy()
semantic_pairwise_comments_df["keep_text"] = (
    semantic_pairwise_comments_df["observed_issue"]
    .fillna(semantic_pairwise_comments_df["raw_text"])
    .fillna("")
)
semantic_pairwise_comments_df["keep_text_length"] = semantic_pairwise_comments_df["keep_text"].astype(str).str.len()

comment_length_lookup = semantic_pairwise_comments_df.set_index("comment_id")["keep_text_length"].to_dict()
comment_screen_lookup = semantic_pairwise_comments_df.set_index("comment_id")["screen_id"].to_dict()

pairwise_decision_rows = []
comments_to_drop = set()

for row in high_similarity_semantic_pairs_df.itertuples(index=False):
    left_id = row.left_comment_id
    right_id = row.right_comment_id
    left_len = int(comment_length_lookup.get(left_id, 0))
    right_len = int(comment_length_lookup.get(right_id, 0))

    if left_len > right_len:
        keep_id, drop_id = left_id, right_id
        decision_reason = "left_longer_text"
    elif right_len > left_len:
        keep_id, drop_id = right_id, left_id
        decision_reason = "right_longer_text"
    else:
        keep_id, drop_id = sorted([left_id, right_id])
        decision_reason = "tie_break_comment_id"

    comments_to_drop.add(drop_id)
    pairwise_decision_rows.append({
        "pair_id": row.pair_id,
        "screen_id": row.screen_id,
        "left_comment_id": left_id,
        "right_comment_id": right_id,
        "left_keep_text_length": left_len,
        "right_keep_text_length": right_len,
        "keep_comment_id": keep_id,
        "drop_comment_id": drop_id,
        "decision_reason": decision_reason,
        "cosine_similarity": row.cosine_similarity,
    })

semantic_pairwise_decisions_df = pd.DataFrame(pairwise_decision_rows)
comments_to_drop_df = pd.DataFrame(
    sorted(
        {
            (comment_id, comment_screen_lookup.get(comment_id))
            for comment_id in comments_to_drop
        },
        key=lambda x: (x[1], x[0]),
    ),
    columns=["comment_id", "screen_id"],
) if comments_to_drop else pd.DataFrame(columns=["comment_id", "screen_id"])

semantic_kept_comments_df = semantic_pairwise_comments_df[
    ~semantic_pairwise_comments_df["comment_id"].isin(comments_to_drop)
].copy()

semantic_comment_payload_cols = [
    "comment_label",
    "missing_parts",
    "expected_standard",
    "observed_issue",
    "suggested_fix",
    "bounding_box",
    "raw_text",
]

semantic_dedup_comments_df = (
    semantic_kept_comments_df
    .sort_values(["screen_id", "screen_task_id", "comment_label"], kind="stable")
    .groupby(["screen_id", "screen_task_id", "task"], sort=True)[semantic_comment_payload_cols]
    .apply(lambda x: x.to_dict(orient="records"))
    .reset_index(name="comments")
)

semantic_task_metadata_df = (
    final_human_reference_df[["screen_id", "app_category", "screen_task_id", "task"]]
    .drop_duplicates()
    .sort_values(["screen_id", "screen_task_id"], kind="stable")
    .reset_index(drop=True)
)

semantic_dedup_comments_df = (
    semantic_task_metadata_df.merge(
        semantic_dedup_comments_df,
        on=["screen_id", "screen_task_id", "task"],
        how="left",
    )
    .sort_values(["screen_id", "screen_task_id"], kind="stable")
    .reset_index(drop=True)
)
semantic_dedup_comments_df["comments"] = semantic_dedup_comments_df["comments"].apply(lambda x: x if isinstance(x, list) else [])
semantic_dedup_comments_df = semantic_dedup_comments_df[semantic_dedup_comments_df["comments"].apply(len) > 0].copy().reset_index(drop=True)
semantic_dedup_comments_df = semantic_dedup_comments_df[["screen_id", "app_category", "screen_task_id", "task", "comments"]]

print("Pairs above threshold:", len(high_similarity_semantic_pairs_df))
print("Unique comments dropped by pairwise rule:", len(comments_to_drop_df))
print("Comments before semantic dedup:", len(semantic_comments_df))
print("Comments after semantic dedup:", len(semantic_kept_comments_df))
print("Task rows after semantic dedup:", len(semantic_dedup_comments_df))

display(semantic_pairwise_decisions_df.head(5))
display(comments_to_drop_df.head(5))
display(semantic_kept_comments_df.drop(columns=["comment_label"], errors="ignore").head(5))

Pairs above threshold: 87
Unique comments dropped by pairwise rule: 86
Comments before semantic dedup: 8203
Comments after semantic dedup: 8117
Task rows after semantic dedup: 1987


,pair_id,screen_id,left_comment_id,right_comment_id,left_keep_text_length,right_keep_text_length,keep_comment_id,drop_comment_id,decision_reason,cosine_similarity
0,PAIR_0034021,71747,H_008169,H_008173,52,51,H_008169,H_008173,left_longer_text,0.988805
1,PAIR_0006253,12175,H_001477,H_001478,77,69,H_001477,H_001478,left_longer_text,0.985627
2,PAIR_0016088,34546,H_003877,H_003879,46,44,H_003877,H_003879,left_longer_text,0.981300
3,PAIR_0008343,16484,H_001989,H_001992,116,103,H_001989,H_001992,left_longer_text,0.979302
4,PAIR_0012669,27321,H_003052,H_003054,81,102,H_003054,H_003052,right_longer_text,0.978443


,comment_id,screen_id
0,H_000072,422
1,H_000074,422
2,H_000185,1369
3,H_000189,1371
4,H_000228,1741


,comment_id,screen_id,app_category,screen_task_id,task,source_type,expected_standard,observed_issue,suggested_fix,match_text,missing_parts,bounding_box,raw_text,keep_text,keep_text_length
0,H_000001,15,Health & Fitness,15_T01,Plan and Start Full Body Workouts,human,the text’s visual treatment and formatting sho...,the text (Workouts) is small even when it is h...,increase font size and weight to make it look...,the text (Workouts) is small even when it is h...,[],"{'x_max': 0.40545597, 'x_min': 0.1559446, 'y_m...",Comment 1 The expected standard is that the te...,the text (Workouts) is small even when it is h...,53
1,H_000002,15,Health & Fitness,15_T01,Plan and Start Full Body Workouts,human,the text and background colors used in the des...,text (Plans) is in light red color on white ba...,change colors to be more complementary to each...,text (Plans) is in light red color on white ba...,[],"{'x_max': 0.1559446, 'x_min': 0.02896114, 'y_m...",Comment 2 The expected standard is that the te...,text (Plans) is in light red color on white ba...,92
2,H_000003,15,Health & Fitness,15_T01,Plan and Start Full Body Workouts,human,the design should make the most important info...,the back button size is small.,increase the back button size to make it visua...,the back button size is small.,[],"{'x_max': 0.1002501, 'x_min': 0.03787226, 'y_m...",Comment 3 The expected standard is that the de...,the back button size is small.,30
3,H_000004,15,Health & Fitness,15_T01,Plan and Start Full Body Workouts,human,design should be well organized.,the texts are disappearing at the bottom edge ...,redesign the UI to fit the elements within the...,the texts are disappearing at the bottom edge ...,[],"{'x_max': 0.75029596, 'x_min': 0.05138287, 'y_...",Comment 4 The expected standard is that design...,the texts are disappearing at the bottom edge ...,157
4,H_000005,15,Health & Fitness,15_T01,Plan and Start Full Body Workouts,human,the design elements should be clearly visible,Icon is not clearly visible,change icon color to dark color to make it cle...,Icon is not clearly visible,[],"{'x_max': 0.90002314, 'x_min': 0.81536749, 'y_...",Comment 5 The expected standard is that the de...,Icon is not clearly visible,27


### 2.4.1 Review One Linked Group And Optionally Drop Kept Comments

Pick one linked-group index, inspect the comments that still remain after the pairwise rule, then optionally remove any of those kept comments in the next cell.


In [7]:
linked_group_idx = 0  # replace with the group you want to inspect

if linked_group_idx < 0 or linked_group_idx >= len(linked_pair_group_summary_df):
    raise IndexError(f"linked_group_idx must be between 0 and {len(linked_pair_group_summary_df) - 1}")

group_row = linked_pair_group_summary_df.iloc[linked_group_idx]
selected_linked_comment_ids = set(group_row.comment_ids)

selected_linked_kept_comments_df = (
    semantic_kept_comments_df[
        semantic_kept_comments_df["comment_id"].isin(selected_linked_comment_ids)
    ]
    .copy()
    .assign(match_text=lambda df: df["observed_issue"].fillna(df["match_text"]))
    .drop(columns=["comment_label"], errors="ignore")
    .sort_values(["screen_task_id", "comment_id"], kind="stable")
    .reset_index(drop=True)
)

print("linked_group_idx:", linked_group_idx)
print("screen_id:", group_row.screen_id)
print("screen_task_id:", group_row.screen_task_id)
print("num_comments_in_group:", group_row.num_comments)
print("num_pairs_in_group:", group_row.num_pairs)
print("kept_comments_in_group_after_pairwise_rule:", len(selected_linked_kept_comments_df))

display(
    selected_linked_kept_comments_df[[
        "comment_id",
        "screen_id",
        "screen_task_id",
        "source_type",
        "match_text",
    ]]
)


linked_group_idx: 0
screen_id: 16484
screen_task_id: 16484_T01
num_comments_in_group: 3
num_pairs_in_group: 3
kept_comments_in_group_after_pairwise_rule: 1


,comment_id,screen_id,screen_task_id,source_type,match_text
0,H_001990,16484,16484_T01,human,the text is difficult to read because it is to...


In [8]:
# Add comment IDs from the displayed table above if you want to remove them.
manual_delete_comment_ids = [
    # "H_000000",
]

manual_delete_comment_ids = list(dict.fromkeys(manual_delete_comment_ids))
selected_kept_comment_ids = set(selected_linked_kept_comments_df["comment_id"])
invalid_manual_delete_ids = sorted(set(manual_delete_comment_ids) - selected_kept_comment_ids)
if invalid_manual_delete_ids:
    raise ValueError(
        "All manual_delete_comment_ids must come from the kept comments shown above. Invalid IDs: "
        + ", ".join(invalid_manual_delete_ids)
    )

selected_manual_delete_comments_df = (
    selected_linked_kept_comments_df[
        selected_linked_kept_comments_df["comment_id"].isin(manual_delete_comment_ids)
    ]
    .copy()
    .sort_values(["screen_task_id", "comment_id"], kind="stable")
    .reset_index(drop=True)
)

semantic_kept_comments_after_manual_delete_df = semantic_kept_comments_df[
    ~semantic_kept_comments_df["comment_id"].isin(manual_delete_comment_ids)
].copy()

print("Manual delete count:", len(manual_delete_comment_ids))
print("Comments after manual delete:", len(semantic_kept_comments_after_manual_delete_df))

display(selected_manual_delete_comments_df[[
    "comment_id",
    "screen_id",
    "screen_task_id",
    "source_type",
    "match_text",
]])


Manual delete count: 0
Comments after manual delete: 8117


,comment_id,screen_id,screen_task_id,source_type,match_text


### 2.4.2 Save Embedding Checkpoints

Save the embedding-stage artifacts needed to resume Section 2.5 without rerunning Sections 2.1 to 2.4.


In [9]:
SEMANTIC_EMBEDDING_DEDUP_PATH = DATA_DIR / "human_comments_semantic_embedding_dedup.parquet"
SEMANTIC_SCORED_PAIRS_PATH = OUTPUT_DIR / "semantic_scored_pairs.parquet"

semantic_dedup_comments_df.to_parquet(SEMANTIC_EMBEDDING_DEDUP_PATH, index=False)
scored_semantic_pairs_df.to_parquet(SEMANTIC_SCORED_PAIRS_PATH, index=False)

print("Saved embedding dedup output:")
print(SEMANTIC_EMBEDDING_DEDUP_PATH)
print("Saved scored semantic pairs:")
print(SEMANTIC_SCORED_PAIRS_PATH)


Saved embedding dedup output:
cleaned_dataset\human_comments_semantic_embedding_dedup.parquet
Saved scored semantic pairs:
cleaned_dataset\human_critiques\semantic_scored_pairs.parquet


### 2.5 OpenAI Batch Flow

Process all uncertain pairs in one batch file, then review the returned decisions before finalizing the semantic-dedup outcome.


In [10]:
SEMANTIC_EMBEDDING_DEDUP_PATH = DATA_DIR / "human_comments_semantic_embedding_dedup.parquet"
SEMANTIC_SCORED_PAIRS_PATH = OUTPUT_DIR / "semantic_scored_pairs.parquet"

if "scored_semantic_pairs_df" not in globals():
    scored_semantic_pairs_df = pd.read_parquet(SEMANTIC_SCORED_PAIRS_PATH)
    print("Loaded scored semantic pairs from:", SEMANTIC_SCORED_PAIRS_PATH)

if "semantic_dedup_comments_df" not in globals() and SEMANTIC_EMBEDDING_DEDUP_PATH.exists():
    semantic_dedup_comments_df = pd.read_parquet(SEMANTIC_EMBEDDING_DEDUP_PATH)
    print("Loaded embedding dedup output from:", SEMANTIC_EMBEDDING_DEDUP_PATH)

if "SEMANTIC_SIMILARITY_THRESHOLD" not in globals():
    SEMANTIC_SIMILARITY_THRESHOLD = 0.90

BATCH_REVIEW_LOWER_BOUND = 0.50
BATCH_MODEL = DEFAULT_REVIEW_MODEL
SEMANTIC_BATCH_REQUESTS_PATH = OUTPUT_DIR / "semantic_duplicate_batch_requests.jsonl"
SEMANTIC_BATCH_RESULTS_BASENAME = "semantic_duplicate_batch_results"

semantic_review_candidates_df = scored_semantic_pairs_df[
    (scored_semantic_pairs_df["cosine_similarity"] >= BATCH_REVIEW_LOWER_BOUND)
    & (scored_semantic_pairs_df["cosine_similarity"] < SEMANTIC_SIMILARITY_THRESHOLD)
].copy().sort_values(["screen_task_id", "cosine_similarity"], ascending=[True, False], kind="stable").reset_index(drop=True)

semantic_review_candidates_df["custom_id"] = semantic_review_candidates_df.apply(
    lambda row: f"semantic_pair::{row['left_comment_id']}::{row['right_comment_id']}",
    axis=1,
)

semantic_batch_requests = build_semantic_review_requests(
    semantic_review_candidates_df,
    custom_id_col="custom_id",
    left_id_col="left_comment_id",
    right_id_col="right_comment_id",
    screen_id_col="screen_id",
    left_comment_col="left_observed_issue",
    right_comment_col="right_observed_issue",
    screen_task_id_col="screen_task_id",
    left_source_col="left_source_type",
    right_source_col="right_source_type",
    model=BATCH_MODEL,
)

write_jsonl(semantic_batch_requests, SEMANTIC_BATCH_REQUESTS_PATH)

print("Batch candidate pairs:", len(semantic_review_candidates_df))
print("Requests written:", len(semantic_batch_requests))
print("Batch request path:", SEMANTIC_BATCH_REQUESTS_PATH)
display(build_standard_preview_df(semantic_review_candidates_df, include_score=True, limit=10))


Batch candidate pairs: 3751
Requests written: 3751
Batch request path: cleaned_dataset\human_critiques\semantic_duplicate_batch_requests.jsonl


,screen_id,screen_task_id,left_observed_issue,right_observed_issue,cosine_similarity
0,10089,10089_T02,the text font size is small and a little diffi...,the text font size is small.,0.883138
1,10089,10089_T02,the text font size is small and a little diffi...,the text is difficult to read because it is to...,0.856498
2,10089,10089_T02,the background makes the foreground text diffi...,the text is difficult to read because it is to...,0.843622
3,10089,10089_T02,the text font size is small.,the text is difficult to read because it is to...,0.733571
4,10089,10089_T02,the background makes the foreground text diffi...,the text font size is small and a little diffi...,0.650007
5,10089,10089_T02,the background makes the foreground text diffi...,the text font size is small.,0.599334
6,10089,10089_T02,the Menu option is not visually prominent.,the arrow key buttons are not visually prominent.,0.563766
7,10089,10089_T02,the Menu option is not visually prominent.,the menu bar is aligned to the left side of th...,0.539695
8,10089,10089_T03,texts (ACC Championship) are in white color on...,the text is difficult to read because it is to...,0.727410
9,10089,10089_T03,"the elements on the page are not aligned, whic...","there are too many elements on the page, which...",0.726224


In [11]:
if semantic_review_candidates_df.empty:
    semantic_batch_requests = []
    print("No semantic-review candidate pairs to send for adjudication.")

View Prompt

In [12]:
PROMPT_PREVIEW_REQUEST_INDEX = 0

if not semantic_batch_requests:
    print("No semantic-review candidates available for prompt preview.")
else:
    if not 0 <= PROMPT_PREVIEW_REQUEST_INDEX < len(semantic_batch_requests):
        raise IndexError(
            f"PROMPT_PREVIEW_REQUEST_INDEX must be between 0 and {len(semantic_batch_requests) - 1}"
        )

    preview_request = semantic_batch_requests[PROMPT_PREVIEW_REQUEST_INDEX]

    preview_messages = preview_request["body"]["messages"]
    print("selected request index:", PROMPT_PREVIEW_REQUEST_INDEX)
    print("custom_id:", preview_request["custom_id"])
    print("\nSystem prompt:\n")
    print(preview_messages[0]["content"])
    print("\nUser prompt:\n")
    print(preview_messages[1]["content"])


selected request index: 0
custom_id: semantic_pair::H_001225::H_001226

System prompt:

You judge the relationship between two UI critique comments written about the same task. The comments may be exact paraphrases, one may fully contain the other plus extra detail, they may partially overlap, or they may be different. Be conservative about merging and return valid JSON only.

User prompt:

Comment A
- source: human
- observed_issue: the text font size is small and a little difficult to read.

Comment B
- source: human
- observed_issue: the text font size is small.

Decide the relationship using one of:
- duplicate: same issue, same meaning, different wording at most
- a_contains_b: A includes B's meaning plus extra detail
- b_contains_a: B includes A's meaning plus extra detail
- partial_overlap: related and overlapping, but not safe to merge automatically
- different: different issue

Then choose a recommended action:
- keep_A
- keep_B
- keep_both
- manual_review

Then provide a conf

#### 2.5.0 Preview One Batch Prompt

Select a `screen_task_id` and inspect one generated request before running the full batch.


#### 2.5.1 Run Batch

Either reuse a saved batch response file or upload the request file, create a batch, monitor it, and retrieve the output.


In [13]:
USE_SAVED_BATCH_OUTPUT = True
SAVED_BATCH_OUTPUT_PATH = OUTPUT_DIR / "openai_semantic_batch" / "Batch_Responses" / "semantic_duplicate_batch_results.jsonl"

batch_client = OpenAI()
batch_manager = OpenAIBatchManager(batch_client, OUTPUT_DIR / "openai_semantic_batch", "semantic_duplicates")

semantic_batch_input_file_id = None
semantic_batch = None
semantic_batch_id = None
semantic_batch_output_path = None

if USE_SAVED_BATCH_OUTPUT:
    semantic_batch_output_path = SAVED_BATCH_OUTPUT_PATH
    print("Using saved batch output:", semantic_batch_output_path)
elif semantic_batch_requests:
    semantic_batch_input_file_id = batch_manager.upload_file(SEMANTIC_BATCH_REQUESTS_PATH)
    semantic_batch = batch_manager.create_batch(semantic_batch_input_file_id)
    semantic_batch_id = getattr(semantic_batch, "id", None) if semantic_batch else None

print("Batch input file id:", semantic_batch_input_file_id)
print("Batch id:", semantic_batch_id)

Using saved batch output: cleaned_dataset\human_critiques\openai_semantic_batch\Batch_Responses\semantic_duplicate_batch_results.jsonl
Batch input file id: None
Batch id: None


In [14]:
if USE_SAVED_BATCH_OUTPUT:
    semantic_batch_status = None
    print("Skipped batch status check because USE_SAVED_BATCH_OUTPUT=True")
elif semantic_batch_id:
    state, semantic_batch_status = batch_manager.check_batch(semantic_batch_id, verbose=True)
    print("Resolved state:", state)
else:
    semantic_batch_status = None
    print("No batch was created.")

Skipped batch status check because USE_SAVED_BATCH_OUTPUT=True


In [15]:
if USE_SAVED_BATCH_OUTPUT:
    print("Batch results loaded from:", semantic_batch_output_path)
elif semantic_batch_id:
    semantic_batch_output_path = batch_manager.retrieve_batch_output(
        batch_id=semantic_batch_id,
        base_name=SEMANTIC_BATCH_RESULTS_BASENAME,
    )
    print("Batch results saved to:", semantic_batch_output_path)
else:
    print("Batch results path:", semantic_batch_output_path)

Batch results loaded from: cleaned_dataset\human_critiques\openai_semantic_batch\Batch_Responses\semantic_duplicate_batch_results.jsonl


#### 2.5.2 Review Batch Output

Inspect the returned decisions for the full batch before finalizing the semantic-dedup decisions.


In [16]:
REVIEW_CONFIDENCE_THRESHOLD = 0.70

if semantic_batch_output_path and Path(semantic_batch_output_path).exists():
    semantic_batch_results_df = parse_semantic_review_results_jsonl(
        semantic_batch_output_path,
        semantic_review_candidates_df,
        left_id_col="left_comment_id",
        right_id_col="right_comment_id",
        include_candidate_columns=[
            "screen_task_id",
            "task",
            "left_observed_issue",
            "right_observed_issue",
            "cosine_similarity",
        ],
    )

    openai_decision_counts = semantic_batch_results_df["recommended_action"].fillna("missing").value_counts().sort_index()
    print("OpenAI decision counts:")
    print(openai_decision_counts.to_string())
    display(semantic_batch_results_df.head(5))

    review_needed_df = identify_review_needed(
        semantic_batch_results_df,
        confidence_threshold=REVIEW_CONFIDENCE_THRESHOLD,
    )

else:
    print("Batch output file not found yet. Run the retrieval cell first.")

OpenAI decision counts:
recommended_action
keep_A            403
keep_B            542
keep_both        2629
manual_review     177


,custom_id,left_id,right_id,relationship,recommended_action,confidence,screen_task_id,task,left_observed_issue,right_observed_issue,cosine_similarity
0,semantic_pair::H_001225::H_001226,H_001225,H_001226,a_contains_b,keep_A,0.95,10089_T02,Explore the ACC Tournament schedules and match...,the text font size is small and a little diffi...,the text font size is small.,0.883138
1,semantic_pair::H_001225::H_001227,H_001225,H_001227,b_contains_a,keep_B,0.95,10089_T02,Explore the ACC Tournament schedules and match...,the text font size is small and a little diffi...,the text is difficult to read because it is to...,0.856498
2,semantic_pair::H_001222::H_001227,H_001222,H_001227,b_contains_a,keep_B,0.93,10089_T02,Explore the ACC Tournament schedules and match...,the background makes the foreground text diffi...,the text is difficult to read because it is to...,0.843622
3,semantic_pair::H_001226::H_001227,H_001226,H_001227,b_contains_a,keep_B,0.90,10089_T02,Explore the ACC Tournament schedules and match...,the text font size is small.,the text is difficult to read because it is to...,0.733571
4,semantic_pair::H_001222::H_001225,H_001222,H_001225,partial_overlap,keep_both,0.90,10089_T02,Explore the ACC Tournament schedules and match...,the background makes the foreground text diffi...,the text font size is small and a little diffi...,0.650007


#### 2.5.3 Interactive Manual Review

Inspect `manual_review` rows one by one and decide on the spot which action to take.


In [18]:
manual_review_decisions_df = (
    inspect_manual_review_rows(review_needed_df)
    if "review_needed_df" in globals()
    else pd.DataFrame()
)
display(manual_review_decisions_df)


1/177 | row_index=612 | confidence=0.78
--------------------------------------------------------------------------------------------------------------
screen_task_id      : 21267_T01
task                : Write a Facebook post to share with friends
relationship       : partial_overlap
recommended_action : manual_review
--------------------------------------------------------------------------------------------------------------
LEFT TEXT:
there is no button on the page for going to the previous page.
--------------------------------------------------------------------------------------------------------------
RIGHT TEXT:
there is no button on the page for navigating further.
--------------------------------------------------------------------------------------------------------------
Current final decision: None


,custom_id,left_id,right_id,relationship,recommended_action,confidence,screen_task_id,task,left_observed_issue,right_observed_issue,cosine_similarity,final_decision
612,semantic_pair::H_002495::H_002496,H_002495,H_002496,partial_overlap,manual_review,0.78,21267_T01,Write a Facebook post to share with friends,there is no button on the page for going to th...,there is no button on the page for navigating ...,0.843539,NaN
247,semantic_pair::H_001780::H_001781,H_001780,H_001781,partial_overlap,manual_review,0.80,14423_T02,"Swipe to the next item, click the back button,...",the views and share buttons are not visually p...,the highlighted buttons are not visually promi...,0.714632,NaN
412,semantic_pair::H_002069::H_002071,H_002069,H_002071,partial_overlap,manual_review,0.80,17259_T03,Tap to confirm your email address to link to F...,progress bar is showing 10% completion however...,"the progress bar is at 10%, but the user has a...",0.789076,NaN
658,semantic_pair::H_002579::H_002581,H_002579,H_002581,partial_overlap,manual_review,0.80,22950_T01,Choose your preferred option from the menu.,the text is difficult to read because there i...,"the colors are too light, making it difficult ...",0.696986,NaN
482,semantic_pair::H_002228::H_002234,H_002228,H_002234,partial_overlap,manual_review,0.80,18604_T01,View Weather Forecast,the app does not resize properly when the user...,"it is not suitable for all device's screen, th...",0.515464,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
3629,semantic_pair::H_001014::H_001023,H_001014,H_001023,partial_overlap,manual_review,0.90,8343_T01,Input Quantity details for Toaster strudel and...,the texts of the design are not consistent in ...,the colors are too dull and the font is too sm...,0.627909,NaN
2727,semantic_pair::H_006650::H_006651,H_006650,H_006651,partial_overlap,manual_review,0.92,57973_T02,Explore your weekly workload with prioritized ...,the navigation bar is missing.,The current placement of the advertisement cre...,0.528243,NaN
607,semantic_pair::H_000272::H_000273,H_000272,H_000273,partial_overlap,manual_review,0.92,2120_T01,Enter the verification code to log in or reque...,the text font is small.,the text inside the buttons is small.,0.744193,NaN
2594,semantic_pair::H_006294::H_006295,H_006294,H_006295,partial_overlap,manual_review,0.92,54237_T01,Enter Bank name to search,there is no link back to the home/previous page.,there is no button on the page for navigating ...,0.702413,NaN


#### 2.5.4 Finalize Batch Decisions

After reviewing the batch output, combine the model decisions with your interactive manual-review overrides.


In [19]:
if "semantic_batch_results_df" in globals():
    accepted_batch_decisions_df, reviewed_rows_df = finalize_review_decisions(
        semantic_batch_results_df,
        manual_review_decisions_df if "manual_review_decisions_df" in globals() else None,
    )

    print("Rows manually reviewed:", len(reviewed_rows_df))
    display(reviewed_rows_df)
else:
    print("No batch results loaded yet.")

Rows manually reviewed: 0


,custom_id,left_id,right_id,relationship,recommended_action,confidence,screen_task_id,task,left_observed_issue,right_observed_issue,cosine_similarity,final_decision


#### 2.5.5 Defer Manual Review For Now

If you want to postpone interactive checking, convert any remaining `manual_review` decisions to `keep_both` so nothing gets dropped by accident.


In [20]:
if "accepted_batch_decisions_df" not in globals():
    if "semantic_batch_results_df" not in globals():
        raise NameError("Load semantic_batch_results_df first by running section 2.5.")

    accepted_batch_decisions_df, reviewed_rows_df = finalize_review_decisions(
        semantic_batch_results_df,
        manual_review_decisions_df if "manual_review_decisions_df" in globals() else None,
    )

deferred_manual_review_mask = accepted_batch_decisions_df["final_decision"].eq("manual_review")
deferred_manual_review_df = accepted_batch_decisions_df.loc[deferred_manual_review_mask].copy()
accepted_batch_decisions_df.loc[deferred_manual_review_mask, "final_decision"] = "keep_both"
deferred_manual_review_df["final_decision"] = "keep_both"

print("Deferred manual-review rows set to keep_both:", len(deferred_manual_review_df))
print("Final decision counts after deferral:")
print(accepted_batch_decisions_df["final_decision"].value_counts().sort_index().to_string())
display(deferred_manual_review_df.head(10))

Deferred manual-review rows set to keep_both: 177
Final decision counts after deferral:
final_decision
keep_A        403
keep_B        542
keep_both    2806


,custom_id,left_id,right_id,relationship,recommended_action,confidence,screen_task_id,task,left_observed_issue,right_observed_issue,cosine_similarity,final_decision
33,semantic_pair::H_001287::H_001288,H_001287,H_001288,partial_overlap,manual_review,0.86,10364_T02,View today's football match Schedule,there is no link back to the home page.,there is no button on the page for navigating ...,0.549824,keep_both
34,semantic_pair::H_001296::H_001298,H_001296,H_001298,partial_overlap,manual_review,0.90,10551_T01,Check details about Etios Cross car and the Of...,the gray text's low contrast with the backgrou...,"the text is difficult to read, and the colors ...",0.822741,keep_both
41,semantic_pair::H_001310::H_001312,H_001310,H_001312,partial_overlap,manual_review,0.90,10584_T02,Look up a number.,the Menu button is not visually prominent.,the navigation buttons are not visually promin...,0.802995,keep_both
42,semantic_pair::H_001312::H_001314,H_001312,H_001314,partial_overlap,manual_review,0.88,10584_T02,Look up a number.,the navigation buttons are not visually promin...,the app's appearance is rather plain and lacks...,0.528694,keep_both
112,semantic_pair::H_001479::H_001480,H_001479,H_001480,partial_overlap,manual_review,0.90,12175_T03,Select yes or no to continue,the inactive button has low contrast with the ...,the next button has low contrast with the back...,0.703783,keep_both
113,semantic_pair::H_001484::H_001487,H_001484,H_001487,partial_overlap,manual_review,0.90,12178_T03,"Select the birthday and tap ""Next"" to continue...",the text font size is small and quite difficul...,"the date is displayed in a small font size, wh...",0.558715,keep_both
217,semantic_pair::H_001682::H_001684,H_001682,H_001684,partial_overlap,manual_review,0.85,13332_T02,Review card type or switch to weekly breakdown...,the clock icon is not able to convey its usabi...,the large black circle in the center of the sc...,0.523986,keep_both
222,semantic_pair::H_001696::H_001698,H_001696,H_001698,partial_overlap,manual_review,0.90,13565_T01,Browse Houses for Rent.,the icon and label are slightly misaligned.,the elements are not aligned in an organized way.,0.520198,keep_both
246,semantic_pair::H_001779::H_001784,H_001779,H_001784,partial_overlap,manual_review,0.90,14423_T02,"Swipe to the next item, click the back button,...",the highlighted texts are small and the backgr...,"the text is too small and difficult to read, a...",0.741928,keep_both
247,semantic_pair::H_001780::H_001781,H_001780,H_001781,partial_overlap,manual_review,0.80,14423_T02,"Swipe to the next item, click the back button,...",the views and share buttons are not visually p...,the highlighted buttons are not visually promi...,0.714632,keep_both


#### 2.5.6 Rebuild Semantic-Dedup Dataset

Apply the finalized pair decisions back onto the comment table and rebuild the task-level semantic-dedup output that section `2.6` will save.


In [21]:
if "accepted_batch_decisions_df" not in globals():
    raise NameError("Run section 2.5.4 first so accepted_batch_decisions_df is available.")

if accepted_batch_decisions_df["final_decision"].eq("manual_review").any():
    raise ValueError("Some rows still have final_decision='manual_review'. Resolve or defer them before rebuilding the output.")

comments_to_drop = set(
    accepted_batch_decisions_df.loc[
        accepted_batch_decisions_df["final_decision"].eq("keep_A"),
        "right_id",
    ].dropna().tolist()
)
comments_to_drop.update(
    accepted_batch_decisions_df.loc[
        accepted_batch_decisions_df["final_decision"].eq("keep_B"),
        "left_id",
    ].dropna().tolist()
)

semantic_kept_comments_df = semantic_comments_df[
    ~semantic_comments_df["comment_id"].isin(comments_to_drop)
].copy()

semantic_comment_payload_cols = [
    "comment_label",
    "missing_parts",
    "expected_standard",
    "observed_issue",
    "suggested_fix",
    "bounding_box",
    "raw_text",
]

semantic_dedup_comments_df = (
    semantic_kept_comments_df
    .sort_values(["screen_id", "screen_task_id", "comment_label"], kind="stable")
    .groupby(["screen_id", "screen_task_id", "task"], sort=True)[semantic_comment_payload_cols]
    .apply(lambda x: x.to_dict(orient="records"))
    .reset_index(name="comments")
)

semantic_task_metadata_df = (
    final_human_reference_df[["screen_id", "app_category", "screen_task_id", "task"]]
    .drop_duplicates()
    .sort_values(["screen_id", "screen_task_id"], kind="stable")
    .reset_index(drop=True)
)

semantic_dedup_comments_df = (
    semantic_task_metadata_df.merge(
        semantic_dedup_comments_df,
        on=["screen_id", "screen_task_id", "task"],
        how="left",
    )
    .sort_values(["screen_id", "screen_task_id"], kind="stable")
    .reset_index(drop=True)
)
semantic_dedup_comments_df["comments"] = semantic_dedup_comments_df["comments"].apply(lambda x: x if isinstance(x, list) else [])
semantic_dedup_comments_df = semantic_dedup_comments_df[
    semantic_dedup_comments_df["comments"].apply(len) > 0
].copy().reset_index(drop=True)
semantic_dedup_comments_df = semantic_dedup_comments_df[["screen_id", "app_category", "screen_task_id", "task", "comments"]]

print("Unique comments dropped by finalized batch decisions:", len(comments_to_drop))
print("Comments after finalized semantic dedup:", len(semantic_kept_comments_df))
print("Task rows after finalized semantic dedup:", len(semantic_dedup_comments_df))
display(semantic_dedup_comments_df.head(3))

Unique comments dropped by finalized batch decisions: 907
Comments after finalized semantic dedup: 7296
Task rows after finalized semantic dedup: 1987


,screen_id,app_category,screen_task_id,task,comments
0,15,Health & Fitness,15_T01,Plan and Start Full Body Workouts,"[{'comment_label': 'Comment 1', 'missing_parts..."
1,15,Health & Fitness,15_T02,Tap on the Plus icon or explore the Workout Pl...,"[{'comment_label': 'Comment 1', 'missing_parts..."
2,28,Finance,28_T02,Enter details to sign in or click on activate ...,"[{'comment_label': 'Comment 1', 'missing_parts..."


### 2.6 Save Semantic-Dedup Output

Save a second final dataset for the semantic-dedup stage, separate from the exact-dedup output.
Suggested name: `human_comments_semantic_dedup.parquet`.


In [22]:
semantic_dedup_output_path = DATA_DIR / "human_comments_semantic_dedup.parquet"

if "semantic_dedup_comments_df" not in globals():
    raise NameError("Run section 2.5.6 first to build semantic_dedup_comments_df.")

assert semantic_dedup_comments_df.columns.tolist() == ["screen_id", "app_category", "screen_task_id", "task", "comments"]
assert semantic_dedup_comments_df["screen_task_id"].is_unique
assert semantic_dedup_comments_df["comments"].apply(len).gt(0).all()

semantic_dedup_comments_df.to_parquet(semantic_dedup_output_path, index=False)

print("Saved:")
print(semantic_dedup_output_path)

Saved:
cleaned_dataset\human_comments_semantic_dedup.parquet
